In [1]:
!pip install httpx trafilatura spacy pandas
!python -m spacy download en_core_web_sm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.6/132.6 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 837.9/837.9 kB 21.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 318.7/318.7 kB 23.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.7/296.7 kB 26.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 86.4 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [5]:
import json
import time
import trafilatura

SEED_URLS = [
    "https://en.wikipedia.org/wiki/FC_Barcelona",
    "https://en.wikipedia.org/wiki/Real_Madrid_CF",
    "https://en.wikipedia.org/wiki/Manchester_United_F.C.",
    "https://en.wikipedia.org/wiki/Liverpool_F.C.",
    "https://en.wikipedia.org/wiki/FC_Bayern_Munich",
    "https://en.wikipedia.org/wiki/Paris_Saint-Germain_F.C.",
    "https://en.wikipedia.org/wiki/Juventus_F.C.",
    "https://en.wikipedia.org/wiki/Manchester_City_F.C.",
    "https://en.wikipedia.org/wiki/Lionel_Messi",
    "https://en.wikipedia.org/wiki/Cristiano_Ronaldo",
    "https://en.wikipedia.org/wiki/Kylian_Mbapp%C3%A9",
    "https://en.wikipedia.org/wiki/Erling_Haaland",
    "https://en.wikipedia.org/wiki/Neymar",
    "https://en.wikipedia.org/wiki/Robert_Lewandowski",
    "https://en.wikipedia.org/wiki/Luka_Modri%C4%87",
    "https://en.wikipedia.org/wiki/Kevin_De_Bruyne",
    "https://en.wikipedia.org/wiki/UEFA_Champions_League",
    "https://en.wikipedia.org/wiki/La_Liga",
    "https://en.wikipedia.org/wiki/Premier_League",
    "https://en.wikipedia.org/wiki/Serie_A",
    "https://en.wikipedia.org/wiki/Ligue_1",
    "https://en.wikipedia.org/wiki/Bundesliga",
    "https://en.wikipedia.org/wiki/Ballon_d%27Or",
    "https://en.wikipedia.org/wiki/FIFA_World_Cup",
    "https://en.wikipedia.org/wiki/2022_FIFA_World_Cup",
    "https://en.wikipedia.org/wiki/Association_football",
    "https://en.wikipedia.org/wiki/Transfer_(association_football)",
]

results = []
for i, url in enumerate(SEED_URLS):
    print(f"[{i+1}/{len(SEED_URLS)}] {url.split('/wiki/')[-1]}...", end=" ")

    try:
        downloaded = trafilatura.fetch_url(url)
        text = trafilatura.extract(downloaded, include_comments=False, include_tables=False)

        if text and len(text.split()) > 100:
            record = {
                "url": url,
                "title": url.split("/wiki/")[-1].replace("_", " ").replace("%C3%A9", "é").replace("%C4%87", "ć").replace("%27", "'"),
                "word_count": len(text.split()),
                "text": text
            }
            results.append(record)
            print(f"OK — {record['word_count']} mots")
        else:
            print("TROP COURT")
    except Exception as e:
        print(f"ERREUR: {e}")

    time.sleep(2)

print(f"\nTotal articles crawlés : {len(results)}")
print(f"Total mots : {sum(r['word_count'] for r in results)}")

[1/27] FC_Barcelona... OK — 19587 mots
[2/27] Real_Madrid_CF... OK — 22466 mots
[3/27] Manchester_United_F.C.... OK — 15614 mots
[4/27] Liverpool_F.C.... OK — 11021 mots
[5/27] FC_Bayern_Munich... OK — 15267 mots
[6/27] Paris_Saint-Germain_F.C.... OK — 11051 mots
[7/27] Juventus_F.C.... OK — 19909 mots
[8/27] Manchester_City_F.C.... OK — 12423 mots
[9/27] Lionel_Messi... OK — 24463 mots
[10/27] Cristiano_Ronaldo... OK — 26099 mots
[11/27] Kylian_Mbapp%C3%A9... OK — 21310 mots
[12/27] Erling_Haaland... OK — 15050 mots
[13/27] Neymar... OK — 39689 mots
[14/27] Robert_Lewandowski... OK — 23183 mots
[15/27] Luka_Modri%C4%87... OK — 28607 mots
[16/27] Kevin_De_Bruyne... OK — 16126 mots
[17/27] UEFA_Champions_League... OK — 8161 mots
[18/27] La_Liga... OK — 6588 mots
[19/27] Premier_League... OK — 19292 mots
[20/27] Serie_A... OK — 5210 mots
[21/27] Ligue_1... OK — 4551 mots
[22/27] Bundesliga... OK — 8449 mots
[23/27] Ballon_d%27Or... OK — 4253 mots
[24/27] FIFA_World_Cup... OK — 9857 mots


In [6]:
with open("crawler_output.jsonl", "w", encoding="utf-8") as f:
    for r in results:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")

print(f"Saved: crawler_output.jsonl ({len(results)} articles)")

Saved: crawler_output.jsonl (27 articles)


In [7]:
import spacy
from collections import Counter

nlp = spacy.load("en_core_web_sm")

ALLOWED_LABELS = {"PERSON", "ORG", "GPE", "DATE", "EVENT", "NORP"}

all_entities = []

for record in results:
    doc = nlp(record["text"][:50000])

    for ent in doc.ents:
        if ent.label_ in ALLOWED_LABELS:
            all_entities.append({
                "text": ent.text,
                "label": ent.label_,
                "source_url": record["url"]
            })

print(f"Entities extracted: {len(all_entities)}")
print(f"\nBy type:")
for label, count in Counter(e["label"] for e in all_entities).most_common():
    print(f"  {label:8s}: {count}")

Entities extracted: 26068

By type:
  DATE    : 8118
  ORG     : 6657
  PERSON  : 5191
  GPE     : 4015
  NORP    : 1266
  EVENT   : 821


In [8]:
import pandas as pd

df = pd.DataFrame(all_entities)
df.to_csv("extracted_entities.csv", index=False)
print(f"Saved: extracted_entities.csv ({len(df)} rows)")

print("\n--- Top PERSON ---")
print(df[df.label == "PERSON"]["text"].value_counts().head(10))
print("\n--- Top ORG ---")
print(df[df.label == "ORG"]["text"].value_counts().head(10))
print("\n--- Top GPE ---")
print(df[df.label == "GPE"]["text"].value_counts().head(10))

Saved: extracted_entities.csv (26068 rows)

--- Top PERSON ---
text
Messi                268
Ronaldo              193
Mbappé               169
Lewandowski          141
Manchester United    126
De Bruyne             64
La Liga               63
BBC Sport             63
Neymar                53
Santos                36
Name: count, dtype: int64

--- Top ORG ---
text
Real Madrid             269
Champions League        203
Juventus                201
FIFA                    170
Bayern                  166
UEFA                    164
Neymar                  135
Liverpool               133
the Champions League    119
PSG                     105
Name: count, dtype: int64

--- Top GPE ---
text
Barcelona          485
Manchester City    142
Haaland            139
Bundesliga         139
Madrid             133
France             106
Liverpool          101
England            100
Modrić              98
Spain               93
Name: count, dtype: int64


In [9]:
def extract_relations(text):
    doc = nlp(text[:50000])
    relations = []

    for sent in doc.sents:
        sent_ents = [ent for ent in sent.ents if ent.label_ in ALLOWED_LABELS]
        if len(sent_ents) < 2:
            continue

        for token in sent:
            if token.dep_ == "ROOT" and token.pos_ == "VERB":
                subject = None
                obj = None
                for child in token.children:
                    if child.dep_ == "nsubj":
                        for ent in sent_ents:
                            if child.idx >= ent.start_char and child.idx < ent.end_char:
                                subject = ent
                    if child.dep_ in ("dobj", "attr", "prep"):
                        for ent in sent_ents:
                            if child.idx >= ent.start_char and child.idx < ent.end_char:
                                obj = ent

                if subject and obj:
                    relations.append({
                        "subject": subject.text,
                        "subject_type": subject.label_,
                        "predicate": token.text,
                        "object": obj.text,
                        "object_type": obj.label_
                    })
    return relations

all_relations = []
for record in results:
    rels = extract_relations(record["text"])
    all_relations.extend(rels)

print(f"Relations extracted: {len(all_relations)}")
for r in all_relations[:15]:
    print(f"  ({r['subject']}) --[{r['predicate']}]--> ({r['object']})")

Relations extracted: 140
  (Gamper) --[recruited]--> (Jack Greenwell)
  (Barcelona) --[won]--> (Campionats de Catalunya)
  (Primo de Rivera) --[jeered]--> (March)
  (Barcelona) --[won]--> (Spanish League)
  (Barcelona) --[beat]--> (Real Madrid 1–0)
  (Barcelona) --[won]--> (the Copa del Rey)
  (Barcelona) --[finished]--> (the 2007–08 season third)
  (Guardiola) --[sold]--> (Ronaldinho)
  (Barça) --[finished]--> (the season)
  (Barcelona) --[won]--> (Club World Cup.[111)
  (Barcelona) --[defeated]--> (Manchester United)
  (Barcelona) --[won]--> (the Club World Cup)
  (Barcelona) --[won]--> (Champions League Final)
  (Quique Setién) --[replaced]--> (Ernesto Valverde)
  (Di Stéfano) --[impressed]--> (Barcelona)


In [10]:
print("=== NER AMBIGUITY CASES ===\n")

print("Case 1: Entity misclassification")
print("Looking for player names detected as ORG or GPE...\n")
known_players = ["Messi", "Ronaldo", "Mbappé", "Neymar", "Haaland"]
for _, row in df.iterrows():
    if any(p in row["text"] for p in known_players) and row["label"] != "PERSON":
        print(f"  '{row['text']}' detected as {row['label']} instead of PERSON")
        break

print("\nCase 2: Club vs City (same name)")
ambiguous = df[df["text"].isin(["Barcelona", "Liverpool", "Munich", "Manchester", "Paris"])]
print(ambiguous.groupby(["text", "label"]).size().reset_index(name="count"))

print("\nCase 3: Ambiguous dates")
dates = df[df.label == "DATE"].head(10)
print(dates[["text", "source_url"]])

=== NER AMBIGUITY CASES ===

Case 1: Entity misclassification
Looking for player names detected as ORG or GPE...

  'Ronaldo' detected as GPE instead of PERSON

Case 2: Club vs City (same name)
          text   label  count
0    Barcelona     GPE    485
1    Barcelona     ORG      6
2    Barcelona  PERSON      4
3    Liverpool     GPE    101
4    Liverpool     ORG    133
5    Liverpool  PERSON      1
6   Manchester     GPE      9
7   Manchester  PERSON      8
8       Munich     GPE     24
9       Munich  PERSON      4
10       Paris     GPE     60
11       Paris  PERSON      1

Case 3: Ambiguous dates
                          text                                  source_url
9                         1899  https://en.wikipedia.org/wiki/FC_Barcelona
19                      annual  https://en.wikipedia.org/wiki/FC_Barcelona
38                        1997  https://en.wikipedia.org/wiki/FC_Barcelona
39                        2009  https://en.wikipedia.org/wiki/FC_Barcelona
40              

In [12]:
from google.colab import files

files.download('crawler_output.jsonl')
files.download('extracted_entities.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>